In [6]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# Silver Layer

## Objective

#Transform the raw Bronze dataset into a clean, validated and standardised Delta table suitable for analytical workloads.

#The Silver layer applies data-quality rules, creates derived business attributes and prepares the data for downstream Gold reporting.

StatementMeta(, 6aba8fa4-6316-46da-b469-cac9ba59f256, 8, Finished, Available, Finished, False)

In [3]:
from pyspark.sql import functions as F

raw_df = spark.read.parquet(
    "Files/green_tripdata_2026-01.parquet"
)

StatementMeta(, 6aba8fa4-6316-46da-b469-cac9ba59f256, 5, Finished, Available, Finished, False)

In [3]:
raw_df.printSchema()

raw_df.count()

StatementMeta(, 1aa4715f-3f2f-4fac-a371-e02c21a5d1c7, 5, Finished, Available, Finished, False)

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- lpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- trip_type: long (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



40272

In [4]:
silver_df = raw_df.dropDuplicates()

StatementMeta(, 6aba8fa4-6316-46da-b469-cac9ba59f256, 6, Finished, Available, Finished, False)

In [5]:
from pyspark.sql import functions as F

silver_df = silver_df.filter(
    F.col("VendorID").isNotNull()
)

StatementMeta(, 6aba8fa4-6316-46da-b469-cac9ba59f256, 7, Finished, Available, Finished, False)

In [9]:
silver_df = silver_df.filter(
    F.col("trip_distance") > 0
)

StatementMeta(, 1aa4715f-3f2f-4fac-a371-e02c21a5d1c7, 11, Finished, Available, Finished, False)

In [11]:
silver_df = silver_df.filter(
    F.col("fare_amount") >= 0
)

StatementMeta(, 1aa4715f-3f2f-4fac-a371-e02c21a5d1c7, 13, Finished, Available, Finished, False)

In [14]:
silver_df = silver_df.filter(
    F.col("lpep_pickup_datetime")
    
    F.col("lpep_dropoff_datetime")
)

StatementMeta(, 1aa4715f-3f2f-4fac-a371-e02c21a5d1c7, 16, Finished, Available, Finished, False)

In [8]:
silver_df = silver_df.withColumn(
    "pickup_date",
    F.to_date("lpep_pickup_datetime")
)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 10, Finished, Available, Finished, False)

In [10]:
silver_df = silver_df.withColumn(
    "pickup_hour",
    F.hour("lpep_pickup_datetime")
)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 12, Finished, Available, Finished, False)

In [12]:
silver_df = silver_df.withColumn(
    "trip_duration_minutes",
    (
        F.unix_timestamp("lpep_dropoff_datetime")
        -
        F.unix_timestamp("lpep_pickup_datetime")
    ) / 60
)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 14, Finished, Available, Finished, False)

In [14]:
silver_df = silver_df.withColumn(
    "revenue_per_mile",
    F.col("total_amount") / F.col("trip_distance")
)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 16, Finished, Available, Finished, False)

In [16]:
from pyspark.sql import functions as F

raw_df = spark.read.parquet(
    "Files/green_tripdata_2026-01.parquet"
)

silver_df = (
    raw_df
    .dropDuplicates()
    .filter(F.col("VendorID").isNotNull())
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("fare_amount") >= 0)
    .filter(F.col("total_amount") >= 0)
    .filter(
        F.col("lpep_pickup_datetime")
        < F.col("lpep_dropoff_datetime")
    )
)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 18, Finished, Available, Finished, False)

In [19]:
silver_df = (
    silver_df
    .withColumn(
        "pickup_date",
        F.to_date("lpep_pickup_datetime")
    )
    .withColumn(
        "pickup_hour",
        F.hour("lpep_pickup_datetime")
    )
    .withColumn(
        "trip_duration_minutes",
        (
            F.unix_timestamp("lpep_dropoff_datetime")
            - F.unix_timestamp("lpep_pickup_datetime")
        ) / 60
    )
    .withColumn(
        "revenue_per_mile",
        F.col("total_amount") / F.col("trip_distance")
    )
)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 21, Finished, Available, Finished, False)

In [21]:
display(
    silver_df.select(
        "lpep_pickup_datetime",
        "lpep_dropoff_datetime",
        "pickup_date",
        "pickup_hour",
        "trip_duration_minutes",
        "revenue_per_mile"
    ).limit(20)
)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3a89b396-1956-4831-b948-9bf4f161dd39)

In [23]:
 bronze_count = raw_df.count()
silver_count = silver_df.count()

print(f"Bronze rows: {bronze_count}")
print(f"Silver rows: {silver_count}")
print(f"Rows removed: {bronze_count - silver_count}")


StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 25, Finished, Available, Finished, False)

Bronze rows: 40272
Silver rows: 38909
Rows removed: 1363


In [25]:
validation_df = silver_df.select(
    F.sum(F.col("VendorID").isNull().cast("int")).alias("NullVendorID"),
    F.sum((F.col("trip_distance") <= 0).cast("int")).alias("InvalidDistance"),
    F.sum((F.col("fare_amount") < 0).cast("int")).alias("NegativeFare"),
    F.sum((F.col("total_amount") < 0).cast("int")).alias("NegativeTotal"),
    F.sum(
        (
            F.col("lpep_pickup_datetime")
            >= F.col("lpep_dropoff_datetime")
        ).cast("int")
    ).alias("InvalidTimestamps")
)

display(validation_df)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0635f4ee-7f2a-44ba-a836-fa4d21dfe7dc)

In [27]:
display(
    silver_df.select(
        "VendorID",
        "pickup_date",
        "pickup_hour",
        "trip_duration_minutes",
        "revenue_per_mile"
    ).limit(20)
)

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 91355106-6f47-4df6-85a1-8e1311e83ce7)

In [30]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_green_taxi_trips")
    

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 32, Finished, Available, Finished, False)

In [32]:
%%sql
SELECT COUNT(*) AS SilverTrips
FROM silver_green_taxi_trips;

StatementMeta(, 22ad1d82-f56a-4fa4-8b44-5378d2960d3e, 34, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
# Silver Layer Summary

## Objective

#Transform the Bronze dataset into a clean, validated and standardised analytical dataset.

## Transformations Applied

#- Removed duplicate records.
#- Removed records with missing VendorID values.
#- Removed invalid trip distances.
#- Removed negative fare values.
#- Validated pickup and drop-off timestamps.
#- Added business-friendly calculated columns:
 #   - pickup_date
 #   - pickup_hour
 #   - trip_duration_minutes
 #   - revenue_per_mile

## Output

#The transformed dataset has been stored as the Delta table:

#silver_green_taxi_trips

#This dataset is now suitable for business modelling within the Gold layer.